### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = ( None )
load_in_4bit = False 
load_in_8bit = False
#unsloth/Llama-3.2-3B-Instruct
#Qwen/Qwen2.5-Coder-3B
#Qwen/Qwen2.5-Coder-7B-Instruct
#Qwen/Qwen2.5-14B-Instruct


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit
)
print(model.dtype)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
torch.bfloat16


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
df=pd.read_csv('svg_score_train1.csv')
df=df[df['score'] > 0.5]
df=df[['topic','svg_code']]

In [6]:
df.shape

(1062, 2)

In [7]:
df1=pd.read_csv('svg_score_first100k.csv')
df1=df1[df1['base_score'] > 0.8]
df1=df1[['caption_blip2','clean_svg_code']]
df1.columns=['topic','svg_code']
print(df1.shape)

(5972, 2)


In [8]:
# df=pd.concat([df,df1.iloc[:3000]],ignore_index=True)
# print(df.shape)

In [9]:
df

,topic,svg_code
0,"'Golden wheat fields under a setting sun',","<svg viewBox=""0 0 200 200"" width=""200"" height=..."
3,"'Snowy mountains under a clear blue sky',","<svg viewBox=""0 0 200 100"" width=""200"" height=..."
4,'Checkerboard pattern with alternating green ...,"<svg viewBox=""0 0 100 100"" width=""100"" height=..."
6,"'Crimson and gold spirals intertwining',","<svg viewBox=""0 0 200 200"" width=""200"" height=..."
7,"'A navy blue trench coat with brass buttons',","<svg viewBox=""0 0 200 300"" width=""200"" height=..."
...,...,...
3075,A quiet wheat field with a scarecrow,"<svg viewBox=""0 0 200 150"" width=""400"" height=..."
3080,A quiet copse,"<svg viewBox=""0 0 200 150"" width=""400"" height=..."
3085,A sleek black tuxedo with a bow tie,"<svg viewBox=""0 0 200 300"" width=""200"" height=..."
3088,A smooth gradient of soft blues,"<svg width=""200"" height=""100"" viewBox=""0 0 200..."


In [16]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["topic"]  # Using 'topic' as instruction
    svgs = examples["svg_code"]  # Using 'svg_code' as output
    texts = []

    instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
    <constraints>
    * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
    * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
    </constraints>
    
    <example>
    <description>"A red circle with a blue square inside"</description>
    ```svg
    <svg viewBox="0 0 256 256" width="256" height="256">
      <circle cx="50" cy="50" r="40" fill="red"/>
      <rect x="30" y="30" width="40" height="40" fill="blue"/>
    </svg>
    ```
    </example>
    
    
    Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. Focus on a clear and concise representation of the input description within the given limitations. Always give the complete SVG code with nothing omitted. Never use an ellipsis.
    
    <description>"{}"</description>
    ```svg
    <svg viewBox="0 0 256 256" width="256" height="256">
    """
    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        instruction= instruction.format(topic)
        text = alpaca_prompt.format(f"Generate a SVG code for the given input:",instruction, svg_code) + EOS_TOKEN
        texts.append(text)

        print(texts)
        
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/1062 [00:00<?, ? examples/s]

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [17]:
# Check dataset sample output
print(dataset["text"][0])

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate a SVG code for the given input:

### Input:
Generate SVG code to visually represent the following text description, while respecting the given constraints.
    <constraints>
    * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
    * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
    </constraints>
    
    <example>
    <description>"A red circle with a blue square inside"</description>
    ```svg
    <svg viewBox="0 0 256 256" width="256" height="256">
      <circle cx="50" cy="50" r="40" fill="red"/>
      <rect x="30" y="30" width="40" height="40" fill="blue

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    #eval_dataset=test_dataset,  # Add test dataset here
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = 1000,
        learning_rate = 5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit", # "adamw_torch" better for fp16
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1062 [00:00<?, ? examples/s]

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,062 | Num Epochs = 8 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 275,251,200/14,000,000,000 (1.97% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.443500
10,0.312400
15,0.228800
20,0.225800
25,0.204600
30,0.214700
35,0.225700
40,0.189600
45,0.189800
50,0.196500


In [ ]:
# #This ONLY saves the LoRA adapters, and not the full model.
# model.save_pretrained("./lora/lora_model_3b_v3") # Local saving
# tokenizer.save_pretrained("./lora/lora_model_3b_v3")

In [ ]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
# model_name = "./lora/lora_model_3b_v2", # YOUR MODEL YOU USED FOR TRAINING
# max_seq_length = 2048,
# dtype = (None),
# load_in_4bit = False,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference


In [ ]:
# # alpaca_prompt = You MUST copy from above!
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Please write a SVG code fo rthe given topic?", # instruction
#         "Golden sun rising in the east", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
# tokenizer.batch_decode(outputs)

In [18]:
#save merged 16bit
model.save_pretrained_merged("./lora/lora4bit_qwen_14B_r64_s1000_i1000_v1", tokenizer, save_method = "merged_16bit",)

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 16.07 out of 31.21 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|███████████████████████████████████████████| 48/48 [02:57<00:00,  3.69s/it]


Unsloth: Saving tokenizer... Done.
Done.
